# Notebook 7: a second multitask network, from CheMeleon

Same 20-head setup as notebook 6, warm-started from `chemprop_chemeleon` instead of
`chemprop_medium`. Run this *after* notebook 6's single seed, not instead of it: the
point is two differently-initialised networks, not one averaged.

**Why CheMeleon, which has failed for us twice.** It failed as a fine-tune on the
challenge labels alone, and again as a frozen embedding feeding TabPFN, where it was
worse than `chemprop_medium` in four cells of four. The first failure now has an
explanation: 1,285 to 2,335 labels drag a pretrained encoder off its pretraining
distribution. Twenty heads over 33,000 molecules remove that failure mode, which is
exactly what `v17` demonstrated with the other checkpoint.

**And it does not need to be good.** `v17`'s arm is *worse* than TabPFN on three of four
endpoints and still earned 0.3 to 0.4 weight, because it correlates 0.72 to 0.92 with the
others where our two TabPFN arms correlate 0.85 to 0.99. A second network from a
different initialisation should be decorrelated from the first for the same reason.

The control matters more here than usual: CheMeleon is a 2,048-wide encoder against
`chemprop_medium`'s 600, so if the checkpoint loads wrong the shapes will still work and
the failure will be silent. Watch the warm-versus-cold line.

About an hour, resumable per fit.

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'NO GPU')

In [ ]:
PAYLOAD_FILE = '/content/drive/MyDrive/cyp_payloads/mini3_d3108f9c6e.tgz'

import os, tarfile
os.makedirs('/content/cache', exist_ok=True)
from google.colab import drive
drive.mount('/content/drive')
with tarfile.open(PAYLOAD_FILE) as t:
    t.extractall('/content/cache')
OUTDIR = '/content/results'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTDIR = '/content/drive/MyDrive/cyp_results'
except Exception as e:
    print('Drive unavailable:', type(e).__name__)
os.makedirs(OUTDIR, exist_ok=True)
print('writing to', OUTDIR)

In [ ]:
!pip -q install chemprop==2.3.1

In [ ]:
import gc, json, time, urllib.request, numpy as np, pandas as pd, torch, torch.nn as nn
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.model_selection import GroupKFold
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')

CACHE=Path('/content/cache'); OUT=Path(OUTDIR)
ISO=['CYP1A2','CYP2C9','CYP2D6','CYP3A4']
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
meta=pd.read_csv(CACHE/'meta_train.csv.gz'); mete=pd.read_csv(CACHE/'meta_test.csv.gz')
hold=np.load(CACHE/'holdout_mask.npy'); G=np.load(CACHE/'scaffold_groups.npy',allow_pickle=True)
AUX=np.load(CACHE/'aux_train.npy')
SM_TR=meta['SMILES'].to_numpy(); SM_TE=mete['SMILES'].to_numpy()

CK='/content/chemprop_chemeleon.pt'
if not Path(CK).exists():
    urllib.request.urlretrieve('https://raw.githubusercontent.com/jeremycheminf/'
        'openadmet_scripts/main/CYP_Challenge/checkpoints/chemprop_chemeleon.pt', CK)
print('checkpoint', round(Path(CK).stat().st_size/1e6,1), 'MB')

URL=('https://pubchem.ncbi.nlm.nih.gov/assay/pcget.cgi?query=download'
     '&record_type=datatable&actvty=all&response_type=save&aid=1851')
if not Path('/content/aid1851.csv').exists():
    urllib.request.urlretrieve(URL, '/content/aid1851.csv')
raw=pd.read_csv('/content/aid1851.csv', low_memory=False, skiprows=[1])
raw['pac50']=-pd.to_numeric(raw['Fit_LogAC50'], errors='coerce')
PANEL={'p450-cyp1a2':'CYP1A2','p450-cyp2c9':'CYP2C9','p450-cyp2c19':'CYP2C19',
       'p450-cyp2d6':'CYP2D6','p450-cyp3a4':'CYP3A4'}
raw['panel']=raw['Panel Name'].map(PANEL)
raw=raw.dropna(subset=['panel','pac50','PUBCHEM_EXT_DATASOURCE_SMILES'])
print('qHTS measurements', len(raw))

CH='/content/cyp_chembl.csv'
if not Path(CH).exists():
    urllib.request.urlretrieve('https://raw.githubusercontent.com/jeremycheminf/'
        'openadmet_scripts/main/CYP_Challenge/data/external/cyp_chembl.csv', CH)
chembl=pd.read_csv(CH)
CH_COLS=[c for c in chembl.columns if c.endswith('_PIC50') and chembl[c].notna().sum()>=200]
print('ChEMBL compounds', chembl.smiles.nunique(), '| heads', CH_COLS)

In [ ]:
def canon(s):
    m=Chem.MolFromSmiles(s) if isinstance(s,str) else None
    return Chem.MolToSmiles(m) if m is not None else None

ours={c for c in map(canon, np.concatenate([SM_TR,SM_TE])) if c}

pub=raw.groupby(['PUBCHEM_EXT_DATASOURCE_SMILES','panel'],as_index=False).pac50.median()
pub['canon']=[canon(s) for s in pub.PUBCHEM_EXT_DATASOURCE_SMILES]
pub=pub.dropna(subset=['canon']); pub=pub[~pub.canon.isin(ours)]
PAN=['CYP1A2','CYP2C9','CYP2C19','CYP2D6','CYP3A4']
wide=pub.pivot_table(index='PUBCHEM_EXT_DATASOURCE_SMILES',columns='panel',
                     values='pac50').reindex(columns=PAN)

ch=chembl.copy()
ch['canon']=[canon(s) for s in ch.smiles]
ch=ch.dropna(subset=['canon']); ch=ch[~ch.canon.isin(ours)]
ch=ch.groupby('canon', as_index=True)[CH_COLS].median()
print('after dropping challenge overlaps: qHTS', len(wide), '| ChEMBL', len(ch))

HEADS=([f'chal_{i}' for i in ISO] + [f'scr_{i}' for i in ISO]
       + [f'qh_{p}' for p in PAN] + [f'ch_{c}' for c in CH_COLS])
n_ch_rows=len(SM_TR); n_pub=len(wide); n_cb=len(ch)
SM_ALL=np.concatenate([SM_TR, wide.index.to_numpy(), ch.index.to_numpy()])
Y=np.full((len(SM_ALL), len(HEADS)), np.nan, np.float32)
for j,i in enumerate(ISO):
    Y[:n_ch_rows, j]=meta[f'{i}_pIC50_direct_inhibition'].to_numpy(float)
    Y[:n_ch_rows, 4+j]=AUX[:, j]
Y[n_ch_rows:n_ch_rows+n_pub, 8:13]=wide.to_numpy(float)
Y[n_ch_rows+n_pub:, 13:]=ch.to_numpy(float)
M=np.isfinite(Y)

# Normalisation from development rows only, never from held-out labels.
_norm=Y.copy(); _norm[np.flatnonzero(hold)]=np.nan
mu=np.nanmean(_norm,0); sd=np.nanstd(_norm,0)
Yz_nan=((Y-mu)/np.where(sd>0,sd,1.0)).astype(np.float32)
print('table', Y.shape, '| labels per head:', dict(zip(HEADS, M.sum(0).tolist())))

In [ ]:
from chemprop import data, featurizers
from chemprop.models.model import MPNN

FZ = featurizers.SimpleMoleculeMolGraphFeaturizer()

# Targets live inside the datapoints, so chemprop's own loader batches them with the
# graphs and nothing has to be re-featurised per minibatch. NaN marks "not measured"
# and becomes the loss mask.
PTS_ALL = [data.MoleculeDatapoint.from_smi(s, Yz_nan[i])
           for i, s in enumerate(SM_ALL)]
PTS_TE = [data.MoleculeDatapoint.from_smi(s, np.zeros(len(HEADS), np.float32))
          for s in SM_TE]
DS_TE = data.MoleculeDataset(PTS_TE, FZ)

def loader(idx=None, bs=128, shuffle=False, test=False):
    pts = PTS_TE if test else ([PTS_ALL[i] for i in idx] if idx is not None else PTS_ALL)
    return data.build_dataloader(data.MoleculeDataset(pts, FZ), num_workers=0,
                                 batch_size=bs, shuffle=shuffle, drop_last=False)

def _fp(m, bmg, V_d, X_d):
    # BatchMolGraph.to() moves in place and returns None, unlike a torch tensor, so it
    # must be called as a statement rather than used as an expression.
    if hasattr(bmg, 'to'):
        bmg.to(DEVICE)
    V_d = V_d.to(DEVICE) if torch.is_tensor(V_d) else V_d
    X_d = X_d.to(DEVICE) if torch.is_tensor(X_d) else X_d
    return m.fingerprint(bmg, V_d, X_d)

class Multi(nn.Module):
    def __init__(self, ckpt, n_out, warm=True):
        super().__init__()
        m = MPNN.load_from_file(ckpt)
        if not warm:
            for p in m.parameters():
                nn.init.xavier_uniform_(p) if p.dim() > 1 else nn.init.zeros_(p)
        self.enc = m
        with torch.no_grad():                      # probe on CPU, before .to(DEVICE)
            bmg, V_d, X_d, *_ = next(iter(loader([0, 1, 2, 3], bs=4)))
            d = m.fingerprint(bmg, V_d, X_d).shape[1]
        self.head = nn.Sequential(nn.Linear(d, 512), nn.ReLU(), nn.Dropout(0.1),
                                  nn.Linear(512, n_out))

    def forward(self, bmg, V_d, X_d):
        return self.head(_fp(self.enc, bmg, V_d, X_d))

def train(tr_idx, epochs=25, warm=True, lr=3e-4, bs=128, seed=0):
    torch.manual_seed(seed)
    m = Multi(CK, len(HEADS), warm).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-5)
    ld = loader(tr_idx, bs=bs, shuffle=True)
    for ep in range(epochs):
        m.train()
        for batch in ld:
            bmg, V_d, X_d, Yb, *_ = batch
            t = Yb.to(DEVICE).float()
            msk = torch.isfinite(t)
            if msk.sum() == 0:
                continue
            opt.zero_grad()
            p = m(bmg, V_d, X_d)
            loss = (((p - torch.nan_to_num(t)) ** 2) * msk).sum() / msk.sum()
            loss.backward()
            opt.step()
    return m

@torch.no_grad()
def predict(m, idx=None, bs=256, test=False):
    m.eval()
    out = [m(*batch[:3]).cpu().numpy() for batch in loader(idx, bs=bs, test=test)]
    return np.vstack(out) * sd + mu

## Control: the warm start still has to beat a cold one

In [ ]:
y1 = meta['CYP3A4_pIC50_direct_inhibition'].to_numpy(float)
obs = np.isfinite(y1)
dev = np.flatnonzero(obs & ~hold); ho = np.flatnonzero(obs & hold)
aux_idx = np.arange(n_ch_rows, len(SM_ALL))      # qHTS and ChEMBL rows
quick = np.concatenate([dev[:800], aux_idx[:4000]])
for warm in [True, False]:
    t0 = time.time()
    m = train(quick, epochs=6, warm=warm)
    p = predict(m, ho)[:, 3]
    print(f'  warm={warm}: CYP3A4 holdout rho {spearmanr(y1[ho], p).statistic:.4f} '
          f'({time.time()-t0:.0f}s for 6 epochs)', flush=True)
    del m; gc.collect(); torch.cuda.empty_cache()

## Full run, resumable per fit

In [ ]:
SEEDS = 1          # one seed; the gain here is a second network, not an average
EPOCHS = 20        # lower to 15 to shave roughly a quarter off

import json
y_all={i:meta[f'{i}_pIC50_direct_inhibition'].to_numpy(float) for i in ISO}
obs_any=np.isfinite(np.column_stack([y_all[i] for i in ISO])).any(1)
dev_rows=np.flatnonzero(obs_any & ~hold); ho_rows=np.flatnonzero(obs_any & hold)
aux_idx=np.arange(n_ch_rows, len(SM_ALL))
folds=list(GroupKFold(5).split(dev_rows,dev_rows,G[dev_rows]))

import pickle
CKPT = OUT / 'nb7_state.pkl'
OLD = OUT / 'nb7_state.npz'
acc, done = {}, set()
if CKPT.exists():
    st = pickle.load(open(CKPT, 'rb')); acc = st['acc']; done = st['done']
elif OLD.exists():
    # migrate the earlier numpy checkpoint; its keys survived, only the index did not
    acc = dict(np.load(OLD, allow_pickle=True)['acc'].item())
    acc = {('%s_%s_%s' % k if len(k) == 3 else '%s_%s' % k) if isinstance(k, tuple)
           else k: v for k, v in acc.items()}
    done = set(acc)
    print('migrated', len(done), 'fits from the old checkpoint')
if done:
    print('resuming;', len(done), 'fits already complete')

def save():
    pickle.dump({'acc': acc, 'done': done}, open(CKPT, 'wb'))

t0=time.time()
for seed in range(SEEDS):
    for f,(tr,va) in enumerate(folds):
        key = f'fold_{f}_{seed}'
        if key in done: continue
        m=train(np.concatenate([dev_rows[tr], aux_idx]), epochs=EPOCHS, seed=seed)
        p=predict(m, dev_rows[va])[:,:4]
        acc[key]=p; done.add(key); save()
        print(f'  seed {seed} fold {f} ({time.time()-t0:.0f}s)', flush=True)
        del m; gc.collect(); torch.cuda.empty_cache()
    for key, idx, kw in [(f'hold_{seed}', ho_rows, {}), (f'test_{seed}', None, {'test': True})]:
        if key in done: continue
        rows = np.concatenate([dev_rows, aux_idx]) if key.startswith('hold') else \
             np.concatenate([dev_rows, ho_rows, aux_idx])
        m=train(rows, epochs=EPOCHS, seed=seed)
        acc[key]=predict(m, idx, **kw)[:,:4]; done.add(key); save()
        print(f'  seed {seed} {key} ({time.time()-t0:.0f}s)', flush=True)
        del m; gc.collect(); torch.cuda.empty_cache()

oof=np.full((n_ch_rows,4), np.nan, np.float32)
for f,(tr,va) in enumerate(folds):
    oof[dev_rows[va]]=np.mean([acc[f'fold_{f}_{s}'] for s in range(SEEDS)],0)
hp=np.mean([acc[f'hold_{s}'] for s in range(SEEDS)],0)
tp=np.mean([acc[f'test_{s}'] for s in range(SEEDS)],0)

RESULTS=[]
for j,i in enumerate(ISO):
    y=y_all[i]
    d=dev_rows[np.isfinite(y[dev_rows])]; h=ho_rows[np.isfinite(y[ho_rows])]
    np.savez_compressed(OUT/f'mtlche_{i}.npz', oof=oof[d,j], dev=d,
                        hold=hp[np.isin(ho_rows,h),j], ho=h, test=tp[:,j])
    r=float(spearmanr(y[d],oof[d,j]).statistic)
    rh=float(spearmanr(y[h],hp[np.isin(ho_rows,h),j]).statistic)
    RESULTS.append(dict(iso=i,dev_spearman=r,holdout_spearman=rh))
    json.dump(RESULTS, open(OUT/'results_nb7.json','w'), indent=2)
    print(f'{i}: dev rho {r:.4f} | holdout rho {rh:.4f}', flush=True)
print('\nnotebook 5, chemprop_medium: dev 0.5577 0.6935 0.4515 0.8022')
print('                              holdout 0.5759 0.6877 0.5529 0.8443')

import shutil
from google.colab import files
shutil.make_archive('/content/notebook7_results','zip',OUT)
files.download('/content/notebook6_results.zip')